# Exploration: RTE `total_forecast`

Notebook d'exploration pour:
- générer un token OAuth RTE depuis un `Basic` base64 éditable,
- appeler l'endpoint `generation_forecast/v3/total_forecast`,
- sur la période **24 mars 2026 -> 28 mars 2026** en **heure française** (`Europe/Paris`).


In [1]:
from __future__ import annotations

import datetime as dt
from zoneinfo import ZoneInfo

import httpx


In [ ]:
# A MODIFIER: colle ici la valeur base64 de "client_id:client_secret"
RTE_BASIC_AUTH_B64 = "PASTE_BASE64"

TOKEN_URL = "https://digital.iservices.rte-france.com/token/oauth/"
TOTAL_FORECAST_URL = "https://digital.iservices.rte-france.com/open_api/generation_forecast/v3/total_forecast"
PARIS_TZ = ZoneInfo("Europe/Paris")

# Optionnel: types de forecast à demander (laisser [] pour ne pas envoyer le paramètre)
TOTAL_FORECAST_TYPES: list[str] = ["D-1", "ID"]

start_date_paris = dt.datetime(2026, 3, 24, 0, 0, 0, tzinfo=PARIS_TZ)
end_date_paris = dt.datetime(2026, 3, 28, 23, 59, 59, tzinfo=PARIS_TZ)

print("start_date_paris:", start_date_paris.isoformat())
print("end_date_paris:  ", end_date_paris.isoformat())


start_date_paris: 2026-03-24T00:00:00+01:00
end_date_paris:   2026-03-28T23:59:59+01:00


In [3]:
if "PASTE_BASE64" in RTE_BASIC_AUTH_B64:
    raise ValueError("Renseigne d'abord RTE_BASIC_AUTH_B64 avec ta vraie valeur base64.")

with httpx.Client(timeout=30.0) as client:
    token_response = client.post(
        TOKEN_URL,
        headers={
            "Authorization": f"Basic {RTE_BASIC_AUTH_B64}",
            "Content-Type": "application/x-www-form-urlencoded",
            "Accept": "application/json",
        },
        data={},
    )

token_response.raise_for_status()
token_payload = token_response.json()
access_token = token_payload.get("access_token")

if not access_token:
    raise RuntimeError(f"Token RTE absent de la reponse: {token_payload}")

print("Token recupere (tronque):", access_token[:12] + "...")


Token recupere (tronque): GnwWvXM49lT8...


In [4]:
params = {
    "start_date": start_date_paris.isoformat(),
    "end_date": end_date_paris.isoformat(),
}
if TOTAL_FORECAST_TYPES:
    params["type"] = ",".join(dict.fromkeys(TOTAL_FORECAST_TYPES))

with httpx.Client(timeout=60.0) as client:
    total_forecast_response = client.get(
        TOTAL_FORECAST_URL,
        headers={
            "Authorization": f"Bearer {access_token}",
            "Accept": "application/json",
        },
        params=params,
    )

total_forecast_response.raise_for_status()
total_forecast_payload = total_forecast_response.json()

print("Status:", total_forecast_response.status_code)
print("Keys:", list(total_forecast_payload.keys()))


Status: 200
Keys: ['total_forecast']


In [27]:
 total_forecast_payload["total_forecast"][0]["points"]

[{'quantity': 2188, 'start_date': '2026-03-23T23:00:00Z'},
 {'quantity': 2165, 'start_date': '2026-03-23T23:15:00Z'},
 {'quantity': 2236, 'start_date': '2026-03-23T23:30:00Z'},
 {'quantity': 2223, 'start_date': '2026-03-23T23:45:00Z'},
 {'quantity': 2117, 'start_date': '2026-03-24T00:00:00Z'},
 {'quantity': 2123, 'start_date': '2026-03-24T00:15:00Z'},
 {'quantity': 2126, 'start_date': '2026-03-24T00:30:00Z'},
 {'quantity': 2142, 'start_date': '2026-03-24T00:45:00Z'},
 {'quantity': 1998, 'start_date': '2026-03-24T01:00:00Z'},
 {'quantity': 2117, 'start_date': '2026-03-24T01:15:00Z'},
 {'quantity': 2186, 'start_date': '2026-03-24T01:30:00Z'},
 {'quantity': 2290, 'start_date': '2026-03-24T01:45:00Z'},
 {'quantity': 2298, 'start_date': '2026-03-24T02:00:00Z'},
 {'quantity': 2225, 'start_date': '2026-03-24T02:15:00Z'},
 {'quantity': 2313, 'start_date': '2026-03-24T02:30:00Z'},
 {'quantity': 2639, 'start_date': '2026-03-24T02:45:00Z'},
 {'quantity': 2583, 'start_date': '2026-03-24T03:00:00Z'

In [26]:
# Apercu rapide des donnees retournees
series = total_forecast_payload["total_forecast"]
print("Nombre de series:", len(series))

if series:
    first_series = series[0]["points"]
    # print("Type:", first_series.get("type"))
    # values = first_series.get("values", [])
    print("Nombre de points:", len(first_series))
    print("Premier point:", first_series[0] if first_series else None)


Nombre de series: 1
Nombre de points: 479
Premier point: {'quantity': 2188, 'start_date': '2026-03-23T23:00:00Z'}


In [ ]:
# Montrer valeurs pour le 24 mars 2026
for dico in total_forecast_payload["total_forecast"][0]["points"]:
    if (dico["start_date"] >= "2026-03-24") & (dico["start_date"] < "2026-03-25"):
        print(dico["start_date"])
        print(dico["quantity"])

# attention, c'est une erreur ici de ne pas traduire en temps Europe/Paris avant de filter

2026-03-24T00:00:00Z
2117
2026-03-24T00:15:00Z
2123
2026-03-24T00:30:00Z
2126
2026-03-24T00:45:00Z
2142
2026-03-24T01:00:00Z
1998
2026-03-24T01:15:00Z
2117
2026-03-24T01:30:00Z
2186
2026-03-24T01:45:00Z
2290
2026-03-24T02:00:00Z
2298
2026-03-24T02:15:00Z
2225
2026-03-24T02:30:00Z
2313
2026-03-24T02:45:00Z
2639
2026-03-24T03:00:00Z
2583
2026-03-24T03:15:00Z
2428
2026-03-24T03:30:00Z
2366
2026-03-24T03:45:00Z
2596
2026-03-24T04:00:00Z
2458
2026-03-24T04:15:00Z
2412
2026-03-24T04:30:00Z
2632
2026-03-24T04:45:00Z
2799
2026-03-24T05:00:00Z
2828
2026-03-24T05:15:00Z
2889
2026-03-24T05:30:00Z
3074
2026-03-24T05:45:00Z
3286
2026-03-24T06:00:00Z
3896
2026-03-24T06:15:00Z
4340
2026-03-24T06:30:00Z
5171
2026-03-24T06:45:00Z
6188
2026-03-24T07:00:00Z
7503
2026-03-24T07:15:00Z
8599
2026-03-24T07:30:00Z
9877
2026-03-24T07:45:00Z
11168
2026-03-24T08:00:00Z
11083
2026-03-24T08:15:00Z
12306
2026-03-24T08:30:00Z
13446
2026-03-24T08:45:00Z
14139
2026-03-24T09:00:00Z
16440
2026-03-24T09:15:00Z
17341
2026-

In [ ]:
dt.datetime.strptime(dico["start_date"], )

TypeError: strptime() takes exactly 2 arguments (1 given)

In [35]:
# Extra: lire la timezone de dico["start_date"] puis convertir vers PARIS_TZ

raw_start_time = dico["start_date"]
parsed_start_time = dt.datetime.fromisoformat(raw_start_time)

source_tz = parsed_start_time.tzinfo
paris_time = parsed_start_time.astimezone(PARIS_TZ)

print("Timezone source:", source_tz)
print("Date convertie Paris:", paris_time.isoformat())
print(dico["start_date"])

Timezone source: UTC
Date convertie Paris: 2026-03-28T23:30:00+01:00
2026-03-28T22:30:00Z


CCL : 
Je lui ai demandé à partir de 2026-03-24T00:00:00+01:00
il m'a renvoyé à partir de 2026-03-23T23:00:00Z, soit 2026-03-23T00:00:00+01:00
C'est normal.
c'est juste qu'il faut convertir à nouveau en temps europe/paris avant de faire des tris
